# EMBER end to end: §3.1 Sparse MF → §3.2 concept features → §3.3 embedding erasure

Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva, EMNLP Findings 2026). This notebook runs the three stages of §3 for one concept,
using the paper's default hyperparameters. Each stage is split into small blocks, and each block starts
with a note on what it does.

| Step | Paper section | What happens | Repo code reused |
|---|---|---|---|
| **3.1** | Finding Embedding Features with Sparse MF | factorize $E_{V^*}^\top \approx ZY$ with WTA-sparse $Y$ | `SparseMatrixFactorization`, `fit_with_ridge` |
| **3.2** | Identifying Concept-Related Features | mass ratio $\rho_i \ge \tau$, then a two-stage Gemini judge → $F_C$ | `compute_embedding_stats`, prompts from `ember/interpret_features.py` |
| **3.3** | Erasing Concept-Related Features | $\mathbf e_t \leftarrow \mathbf e_t - \delta \sum_{i\in F_C} Y_{i,t}\,\mathbf z_i$ for concept tokens | same math as `ember/erasure/embed_edit.py` |

**Notation.** Paper $Z \in \mathbb R^{d\times k}$ = code `F_`; paper $Y \in \mathbb R^{k\times|V^*|}$ = code `G_.T`.
In this notebook, `Z` and `Y` use the paper's shapes.

**Requirements:** a GPU, `HF_TOKEN` (only needed for gated models such as Llama), and a Gemini API key for Step 3.2.

---
## 0. Setup

### 0.1 Hyperparameters (paper defaults)

| Symbol | Value | Source |
|---|---|---|
| $k$ (rank) | 100 for Gemma-2-2B-it, 200 for Llama-3.1-8B-Instruct | App. A.1 |
| $s$ (WTA sparsity on $Y$) | 0.01, so each factor keeps $\lceil 0.01\,|V^*| \rceil$ tokens | App. A.1 |
| $\lambda$ (ridge) | $10^{-4}$ | App. A.1 |
| $T$ / patience / tol | 20,000 iters / 500 iters / $10^{-4}$ | App. A.1 |
| $\|S_C\| = \|S_N\|$ | 300 sentences each (`data/*_sentences.json`) | §4.1 |
| $\tau$ (mass-ratio threshold) | 2.0 | App. A.3 |
| judge confidence | ≥ 0.85, with 20 tokens per feature | App. A.3 |
| $\delta$ (edit strength) | tuned per concept over {0.5, 1, 2, 5, 10, 50, 100, 200} | App. C.3 |

In [1]:
import gc
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # HF_TOKEN / GEMINI_API_KEY from .env, if present
pd.set_option("display.max_colwidth", 200)

# ---- what to erase ------------------------------------------------------
CONCEPT = "Harry Potter"
MODEL_NAME = "google/gemma-2-2b-it"     # or "meta-llama/Llama-3.1-8B-Instruct"
SEED = 42

# ---- Step 3.1: Sparse MF (App. A.1) -------------------------------------
RANK = 200 if "llama" in MODEL_NAME.lower() else 100   # k
SPARSITY = 0.01          # s
RIDGE = 1e-4             # lambda
MAX_ITER = 20_000        # T
PATIENCE = 500

# ---- Step 3.2: feature selection (App. A.3) -----------------------------
TAU = 2.0
CONFIDENCE = 0.85
TOP_K_TOKENS = 20
GEMINI_MODEL = "models/gemini-3.5-flash-lite"   # repo default; the paper used Gemini-2.5-Flash-Lite

# ---- Step 3.3: erasure (App. C.3) ---------------------------------------
DELTA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 200.0]
DELTA = 10.0             # repo demo's pick for Harry Potter / Gemma; use the sweep in 3.3.f for other concepts

concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
assert CONCEPT in concepts, f"{CONCEPT!r} is not one of: {concepts}"
print(f"concept={CONCEPT!r} | model={MODEL_NAME} | k={RANK}")

concept='Harry Potter' | model=google/gemma-2-2b-it | k=100


/homes/orip/.conda/envs/Ember/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 0.2 Gemini API key

Step 3.2 is the only step that calls Gemini. You can paste your key below or put `GEMINI_API_KEY=...`
in `.env`. Clear the key from this cell before you commit or share the notebook.

In [2]:
GEMINI_API_KEY = ""  # paste here, or leave empty to use .env / the environment

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert any(os.getenv(k) for k in ("GOOGLE_API_KEY", "GEMINI_API_TOKEN", "GEMINI_API_KEY")), \
    "No Gemini key found: Step 3.2 needs one."
print("Gemini key found.")

Gemini key found.


### 0.3 Repo imports

These are the building blocks the paper's pipeline uses. Reusing them means the notebook gives the same
results as `ember.train_mf_features`, `ember.interpret_features`, and `ember.run_erasure`.

In [3]:
from llm_utils.activation_generator import ActivationGenerator, extract_token_ids_sample_ids_and_labels
from transformers import AutoTokenizer

from ember.erasure.model_loader import load_hf_model
from ember.evals.mc import evaluate_mc_generation, prepare_mc_items
from ember.evals.model_wrap import WrappedHFModel
from ember.interpret_features import (
    MEMBERSHIP_PROMPT,
    TOKENS_DESC_SYS,
    SimpleGeminiClient,
    classify_membership_with_desc,
    get_description,
)
from ember.local_datasets import ConceptDataset, load_mc_qa_items, load_open_qa_examples
from ember.timing import Timer
from ember.utils import (
    SparseMatrixFactorization,
    _safe_model_name,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    fit_with_ridge,
    get_embedding_matrix,
    get_special_token_ids,
    resolve_device,
    set_seed,
)

set_seed(SEED)
# DEVICE = resolve_device("auto")
# This host's vGPU reports CUDA as available but prohibits compute, so force CPU.
DEVICE = "cpu"
os.environ["EMBER_DEVICE"] = "cpu"  # load_hf_model reads this
print(f"device: {DEVICE}")

/homes/orip/.conda/envs/Ember/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


device: cpu


---
# Step 3.1: Finding embedding features with Sparse Matrix Factorization

**Goal:** split each token embedding $\mathbf e_t$ into a sum of a few shared directions, so that
semantically related tokens (e.g. *Harry*, *Hogwarts*, *wand*) share a feature:

$$E_{V^*}^\top \approx Z\,Y,\qquad Z\in\mathbb R^{d\times k}\ \text{(dense features)},\quad Y\in\mathbb R^{k\times|V^*|}\ \text{(WTA-sparse coefficients)}$$

The hard winner-takes-all (WTA) constraint on the rows of $Y$ means each feature $\mathbf z_i$ is used by only
1% of the tokens in $V^*$. That is why a feature can be specific to one concept.

### 3.1.a Load the model (TransformerLens)

The pipeline reads $E$ through TransformerLens. **Note:** for Gemma, TL folds the $\sqrt{d}$ embedding
multiplier into `W_E`, so the factorization runs in this *scaled* space. Step 3.3 accounts for that
when it writes back into the HF weights.

In [4]:
act_generator = ActivationGenerator(MODEL_NAME, model_device=DEVICE, data_device="cpu", mode="mlp")
tl_model = act_generator.model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

E = get_embedding_matrix(tl_model)  # (|V|, d)
VOCAB_SIZE, D_MODEL = E.shape
print(f"E: {tuple(E.shape)}  dtype={E.dtype}")

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.90it/s]


Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
E: (256000, 2304)  dtype=torch.float32


### 3.1.b Build $S_C \cup S_N$ and the token set $V^*$

- $S_C$: 300 Wikipedia sentences about the concept. $S_N$: 300 neutral Wikipedia sentences.
- $V^*$: the unique non-special token ids in $S_C \cup S_N$. The model factorizes only these rows of $E$,
  which is cheaper and keeps the factorization focused on the concept. The neutral tokens let Step 3.2
  tell concept features apart from general ones.

In [5]:
dataset = ConceptDataset(concept_name=CONCEPT, neutral_sample_seed=42)
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(tl_model)
vstar_ids = sorted({int(t) for t in token_ids_all.tolist()} - special)

n_concept_sents = sum(lbl == CONCEPT for _, lbl in dataset)
print(f"|S_C| = {n_concept_sents}   |S_N| = {len(dataset) - n_concept_sents}")
print(f"token occurrences = {len(token_ids_all)}  ->  |V*| = {len(vstar_ids)} "
      f"({len(vstar_ids) / VOCAB_SIZE:.2%} of the vocabulary)")

Extracting token IDs: 100%|██████████| 120/120 [00:00<00:00, 930.20it/s]

|S_C| = 300   |S_N| = 300
token occurrences = 18611  ->  |V*| = 4780 (1.87% of the vocabulary)


### 3.1.c Build the data matrix $A = E_{V^*}^\top$

Each column of $A$ is one token's embedding.

In [6]:
E_vstar = E.index_select(0, torch.tensor(vstar_ids, device=E.device)).detach().float()  # (|V*|, d)
A = E_vstar.T.contiguous().to(DEVICE)                                                      # (d, |V*|)

print(f"A: {tuple(A.shape)}  (d x |V*|)")
print(f"WTA keeps ceil(s*|V*|) = {math.ceil(SPARSITY * len(vstar_ids))} tokens per feature")

A: (2304, 4780)  (d x |V*|)
WTA keeps ceil(s*|V*|) = 48 tokens per feature


### 3.1.d Fit Sparse MF (alternating least squares + WTA)

The fit alternates two closed-form ridge updates:

$$Z \leftarrow A Y^\top (Y Y^\top + \lambda I)^{-1},\qquad Y \leftarrow (Z^\top Z + \lambda I)^{-1} Z^\top A$$

After each $Y$ update it applies WTA, which keeps the top-$s$ fraction of $|Y_{i,:}|$ and zeroes the rest.
It also rescales between the two factors. Initialization is $\mathcal N(0,1)$, and the fit stops early after
500 iterations without an improvement of more than $10^{-4}$. If the matrix inverse is singular,
`fit_with_ridge` retries with $\lambda \times 10$.

This is the slowest cell in Step 3.1. It prints progress every 50 iterations.

In [7]:
set_seed(SEED)
smf = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
with Timer() as t_fit:
    ridge_used = fit_with_ridge(smf, A, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)

Z = smf.F_.detach().float().cpu()      # (d, k)      embedding features z_i (columns)
Y = smf.G_.detach().float().cpu().T    # (k, |V*|)   sparse coefficients
print(f"\nfitted in {t_fit['elapsed']:.1f}s (lambda={ridge_used:g})  Z: {tuple(Z.shape)}  Y: {tuple(Y.shape)}")

Iter    0: loss=31131044.000000  (best=31131044.000000, no_improve=0)
Iter   50: loss=26738140.000000  (best=26738140.000000, no_improve=0)
Iter   51: loss=26738156.000000  (best=26738140.000000, no_improve=1)
Iter   73: loss=26733192.000000  (best=26733186.000000, no_improve=1)
Iter   76: loss=26733132.000000  (best=26733050.000000, no_improve=1)
Iter   80: loss=26732868.000000  (best=26732848.000000, no_improve=1)
Iter   82: loss=26732864.000000  (best=26732824.000000, no_improve=1)
Iter   87: loss=26732874.000000  (best=26732812.000000, no_improve=1)
Iter   98: loss=26732758.000000  (best=26732728.000000, no_improve=1)
Iter  100: loss=26732812.000000  (best=26732722.000000, no_improve=1)
Iter  109: loss=26731390.000000  (best=26731380.000000, no_improve=1)
Iter  112: loss=26731682.000000  (best=26731274.000000, no_improve=1)
Iter  133: loss=26729652.000000  (best=26729652.000000, no_improve=1)
Iter  145: loss=26729854.000000  (best=26729592.000000, no_improve=1)
Iter  150: loss=2672

### 3.1.e Check the factorization

Reconstruction quality and sparsity. Each feature should use ≈ $\lceil s|V^*|\rceil$ tokens. Tokens that
no feature covers are left as pure reconstruction error $\boldsymbol\varepsilon_t$.

In [8]:
A_cpu = A.cpu()
rel_err = (torch.norm(A_cpu - Z @ Y) / torch.norm(A_cpu)).item()
nz = Y.abs() > 0
tokens_per_feature = nz.sum(1)
features_per_token = nz.sum(0)

print(f"relative reconstruction error ||A - ZY||_F / ||A||_F = {rel_err:.4f}")
print(f"tokens per feature : min={tokens_per_feature.min().item()}  max={tokens_per_feature.max().item()}")
print(f"features per token : mean={features_per_token.float().mean():.2f}  max={features_per_token.max().item()}  "
      f"uncovered tokens={int((features_per_token == 0).sum())}")

relative reconstruction error ||A - ZY||_F / ||A||_F = 0.9158
tokens per feature : min=48  max=48
features per token : mean=1.00  max=5  uncovered tokens=1394


In [9]:
print(Z.size())
print(Y.size())
print(RANK)

torch.Size([2304, 100])
torch.Size([100, 4780])
100


### 3.1.f Baseline: truncated SVD with the same $k$

The rank-$k$ truncated SVD $A_k = U_k \Sigma_k V_k^\top$ is the best possible rank-$k$ approximation
(Eckart–Young theorem), so no rank-$k$ factorization can have a lower error, including Sparse MF.
The gap between the two errors is what the WTA sparsity costs in reconstruction.

In [10]:
U, S, Vh = torch.linalg.svd(A_cpu, full_matrices=False)
A_svd = U[:, :RANK] @ torch.diag(S[:RANK]) @ Vh[:RANK]
svd_rel_err = (torch.norm(A_cpu - A_svd) / torch.norm(A_cpu)).item()

svd_nnz = U[:, :RANK].numel() + RANK + Vh[:RANK].numel()
smf_nnz = Z.numel() + int(nz.sum())
print(f"rank-{RANK} truncated SVD : relative error = {svd_rel_err:.4f}   (parameters: {svd_nnz:,})")
print(f"rank-{RANK} Sparse MF     : relative error = {rel_err:.4f}   (nonzero parameters: {smf_nnz:,})")

# Smallest SVD rank whose error is no worse than Sparse MF's.
tail = torch.sqrt(torch.flip(torch.cumsum(torch.flip(S ** 2, [0]), 0), [0]) / (S ** 2).sum())
matching_rank = int((tail <= rel_err).nonzero()[0]) if (tail <= rel_err).any() else len(S)
print(f"SVD reaches the Sparse MF error ({rel_err:.4f}) at rank {matching_rank}")

rank-100 truncated SVD : relative error = 0.8219   (parameters: 708,500)
rank-100 Sparse MF     : relative error = 0.9158   (nonzero parameters: 235,200)
SVD reaches the Sparse MF error (0.9158) at rank 5


---
# Step 3.2: Identifying concept-related features

Most of the $k$ features are general (punctuation, function words, and so on). The selection runs in two filters:

1. **Mass ratio** $\rho_i$: is feature $i$ used mostly by concept tokens rather than neutral tokens? Keep the feature if $\rho_i \ge \tau = 2.0$.
2. **LLM judge** (Gemini, two stages): describe the feature's tokens *without* seeing the concept name,
   then ask whether that description is the concept. Keep the feature if `is_member` is true and confidence ≥ 0.85.

The features that pass both filters form $F_C$.

### 3.2.a Label every token in $V^*$

$\ell_t \in$ {concept, neutral, both}, depending on whether $t$ appears in $S_C$ only, $S_N$ only, or both.
Codes: `1` = concept, `0` = neutral, `2` = both.

In [11]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
codes = np.array([token_to_code.get(t, 0) for t in vstar_ids])

print(f"T_C (concept-only) = {(codes == 1).sum()}   T_N (neutral-only) = {(codes == 0).sum()}   "
      f"both = {(codes == 2).sum()}")

T_C (concept-only) = 1870   T_N (neutral-only) = 2229   both = 681


### 3.2.b Score features by mass ratio and keep $\rho_i \ge \tau$

As implemented in `compute_embedding_stats` (column `ratio_abs`):

$$\rho_i = \frac{\operatorname{mean}_{t\in T_C} |Y_{i,t}|}{\operatorname{mean}_{t\in T_N} |Y_{i,t}|}$$

Tokens labeled *both* are excluded from the ratio. The threshold sweep below is the per-concept view of
Figure 6 in the paper.

In [12]:
stats = compute_embedding_stats(Y.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)

for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"rho >= {thr:<4}: {(stats['ratio_abs'] >= thr).sum():3d} / {RANK} features")

candidates = stats[stats["ratio_abs"] >= TAU]
print(f"\n{len(candidates)} candidate features pass tau = {TAU}")
candidates[["feature", "ratio_abs", "mean_abs_concept", "mean_abs_neutral", "num_concept", "num_neutral", "num_both"]]

rho >= 1.0 :  65 / 100 features
rho >= 1.25:  55 / 100 features
rho >= 1.5 :  51 / 100 features
rho >= 1.75:  42 / 100 features
rho >= 2.0 :  34 / 100 features
rho >= 2.25:  27 / 100 features

34 candidate features pass tau = 2.0


,feature,ratio_abs,mean_abs_concept,mean_abs_neutral,num_concept,num_neutral,num_both
22,22,25.952965,0.002739,0.000106,42,4,2
9,9,22.973695,0.002181,0.000095,37,3,8
67,67,9.297039,0.002996,0.000322,39,5,4
36,36,7.024655,0.002457,0.000350,35,5,8
73,73,6.973225,0.001962,0.000281,32,7,9
12,12,5.701410,0.002508,0.000440,35,7,6
84,84,5.459760,0.001678,0.000307,26,5,17
98,98,5.032783,0.002441,0.000485,36,8,4
92,92,4.575541,0.002127,0.000465,30,14,4
20,20,4.345542,0.002231,0.000513,31,9,8


### 3.2.c Collect the tokens the judge will see

For each candidate, the code takes the tokens with nonzero $Y_{i,t}$, sorted by $|Y_{i,t}|$, and keeps the top 20.
The paper text says "concept-labeled tokens", but the released code sends the top activating tokens
whatever their label. This notebook follows the code. The `labels` column shows each token's label.

In [13]:
token_label_map = {tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral" for tid, c in token_to_code.items()}
feature_rows = collect_feature_rows_for_embeddings(
    Y.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)
feat_tokens = pd.DataFrame(feature_rows).set_index("feature")[["activating_tokens", "labels"]]

cand = candidates[["feature", "ratio_abs"]].join(feat_tokens, on="feature").reset_index(drop=True)
cand["judge_tokens"] = cand["activating_tokens"].apply(lambda toks: toks[:TOP_K_TOKENS])
cand[["feature", "ratio_abs", "judge_tokens"]]

,feature,ratio_abs,judge_tokens
0,22,25.952965,"[ magic, Magic, magic, Magic, magical, magical, Magical, wizard, wizards, witchcraft, Wizard, witch, Witch, witches, Wizards, wizard, Sorcerer, wiz, fantasy, wand]"
1,9,22.973695,"[ writing, written, writing, written, write, wrote, writers, writer, reading, reads, read, speaking, letters, language, Read, scribe, speak, authors, letter, author]"
2,67,9.297039,"[ learns, teaches, tells, undergoes, goes, assumes, allows, speaks, distinguishes, enters, becomes, explains, gives, takes, weighs, delivers, engages, comes, manages, describes]"
3,36,7.024655,"[ depiction, portrayal, depictions, portray, portraying, depicting, depicted, interpretation, representing, rendition, interpretations, represented, represent, version, translation, ..."
4,73,6.973225,"[ book, books, Book, book, Books, novels, novel, Novel, bookstores, textbook, ebooks, novelist, album, albums, movie, fiction, literature, Libr, fiction, films]"
5,12,5.701410,"[ criticism, criticisms, criticized, criticised, critics, critic, controversy, controversies, debate, argue, argument, accusations, arguing, issues, complains, concerns, argues, p..."
6,84,5.459760,"[ though, however, although, Though, Though, However, therefore, meanwhile, Although, nonetheless, instead, rather, Rather, compared, but, while, thus, Despite, Unlike, Yet]"
7,98,5.032783,"[ love, Love, loves, Love, loved, likes, desire, wants, Wants, hope, dislike, fans, joy, favor, fan, favourite, Hope, Wish, happiness, favour]"
8,92,4.575541,"[ selling, selling, sold, sold, sale, sales, seller, buying, purchasing, marketing, bought, promotion, retail, market, bestselling, promotional, pricing, Vendors, commercial, promot..."
9,20,4.345542,"[ show, display, showing, show, displays, displayed, shown, showed, showcase, showcase, exhibition, presented, depicting, reveal, depicted, portray, illustrate, portraying, present..."


### 3.2.d-0 Rate-limited Gemini client

The free tier allows **15 requests per minute per model**. Stage 1 and Stage 2 make one request per
candidate each, so 2 × (number of candidates) requests in total. This wrapper does three things:

- spaces requests at least `60 / RPM` seconds apart (with a 10% margin);
- on a per-minute 429, waits the `retry in Ns` delay the server reports, then retries;
- fails immediately on a *daily* quota error, because waiting would not help.

Answers are cached in `descriptions` and `verdicts`. Re-running a stage only queries the features that
are still missing, so a crash never wastes quota you already spent.

In [14]:
import re
import time

GEMINI_RPM = 15


class ThrottledGemini:
    def __init__(self, inner: SimpleGeminiClient, rpm: int = GEMINI_RPM, max_attempts: int = 6):
        inner.max_retries = 1  # retries are handled here, with the server's retry delay
        self.inner = inner
        self.interval = 60.0 / rpm * 1.1
        self.max_attempts = max_attempts
        self._last = 0.0

    def generate(self, prompt: str) -> str:
        for attempt in range(self.max_attempts):
            time.sleep(max(0.0, self._last + self.interval - time.monotonic()))
            self._last = time.monotonic()
            try:
                return self.inner.generate(prompt)
            except RuntimeError as e:
                msg = str(e)
                if "429" not in msg or "PerDay" in msg or attempt == self.max_attempts - 1:
                    raise
                m = re.search(r"retry in ([\d.]+)s", msg)
                time.sleep(float(m.group(1)) + 1 if m else 60)


USE_THROTTLE = False  # billing enabled; set True to go back to free-tier pacing

if USE_THROTTLE:
    client = ThrottledGemini(SimpleGeminiClient(model_name=GEMINI_MODEL))
else:
    client = SimpleGeminiClient(model_name=GEMINI_MODEL)

# Kept across re-runs; call .clear() on both after re-fitting Step 3.1.
descriptions = globals().get("descriptions", {})
verdicts = globals().get("verdicts", {})
print(f"{2 * len(cand)} requests needed, {len(descriptions)} descriptions and {len(verdicts)} verdicts cached; "
      f"throttle={'on' if USE_THROTTLE else 'off'}")

68 requests needed, 0 descriptions and 0 verdicts cached; throttle=off


### 3.2.d LLM judge, stage 1: describe each feature *without* the concept name

Stage 1 never sees the concept name. This avoids confirmation bias, where tokens like *wizard* could
look like "Harry Potter" just because the judge was told the concept.

In [15]:
print("---- Stage 1 prompt ----")
print(TOKENS_DESC_SYS.strip(), "\n\nTokens: {tokens}\nExplanation of feature behavior:")

todo = cand[~cand["feature"].astype(int).isin(descriptions)]
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Stage 1: describe"):
    descriptions[int(row["feature"])] = get_description(client, row["judge_tokens"])
cand["description"] = cand["feature"].map(descriptions)
cand[["feature", "ratio_abs", "description"]]

---- Stage 1 prompt ----
You get tokens that represent a single feature vector (with some noise).
Infer the single most specific, cohesive concept shared by the relevant tokens.
Return only one concise sentence. No preface, no list, no caveats.

Example:
Tokens: ['▁tomorrow','▁tonight','▁yesterday','▁today','▁demain']
Explanation of feature behavior: this vector is related to specific dates and times (e.g., today/tomorrow/yesterday). 

Tokens: {tokens}
Explanation of feature behavior:


Stage 1: describe: 100%|██████████| 34/34 [00:23<00:00,  1.46it/s]


,feature,ratio_abs,description
0,22,25.952965,"This vector activates on tokens related to magic, witchcraft, and fantasy."
1,9,22.973695,"This vector represents the concepts of written language, literacy, and the act or creators of writing."
2,67,9.297039,This vector represents active third-person singular present tense verbs expressing various actions and states.
3,36,7.024655,This feature vector represents the concept of depicting or representing something through various forms of interpretation and translation.
4,73,6.973225,"This vector represents long-form narrative media, published literature, and bound reading materials."
5,12,5.701410,"This vector represents the concept of expressing disagreement, debate, and criticism regarding controversial issues."
6,84,5.459760,"These tokens function as transitional adverbs and conjunctions used to indicate contrast, concession, or logical consequence between clauses or sentences."
7,98,5.032783,"This vector represents positive emotion, affection, and personal preferences or desires."
8,92,4.575541,"This feature vector represents the commercial concepts of buying, selling, and retail market transactions."
9,20,4.345542,"This vector represents the concept of visual presentation, exhibition, or depiction."


In [16]:
N_SHOW = 10
LABEL_TAG = {1: "C", 0: "N", 2: "B"}  # concept / neutral / both

for _, row in cand.sort_values("ratio_abs", ascending=False).iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini : {descriptions.get(feat, '<not described yet>')}")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 22  rho=25.95 ===
  Gemini : This vector activates on tokens related to magic, witchcraft, and fantasy.
  tokens : '▁magic'[C] 0.32, 'Magic'[C] 0.31, 'magic'[C] 0.30, '▁Magic'[B] 0.30, '▁magical'[C] 0.26, 'magical'[C] 0.25, '▁Magical'[C] 0.23, '▁wizard'[C] 0.20, '▁wizards'[C] 0.20, '▁witchcraft'[C] 0.18

=== feature 9  rho=22.97 ===
  Gemini : This vector represents the concepts of written language, literacy, and the act or creators of writing.
  tokens : '▁writing'[B] 0.33, '▁written'[B] 0.31, 'writing'[C] 0.31, 'written'[C] 0.30, '▁write'[C] 0.30, '▁wrote'[B] 0.27, '▁writers'[C] 0.23, '▁writer'[B] 0.23, '▁reading'[B] 0.16, '▁reads'[C] 0.16

=== feature 67  rho=9.30 ===
  Gemini : This vector represents active third-person singular present tense verbs expressing various actions and states.
  tokens : '▁learns'[C] 0.18, '▁teaches'[C] 0.17, '▁tells'[C] 0.17, '▁undergoes'[C] 0.17, '▁goes'[B] 0.16, '▁assumes'[C] 0.16, '▁allows'[C] 0.16, '▁speaks'[C] 0.16, '▁distinguishes'[C] 0

### 3.2.e LLM judge, stage 2: is the description *the* concept?

Stage 2 sees only the concept name and the Stage 1 description. It returns `{"is_member", "confidence"}`.
A feature joins $F_C$ only if `is_member` is true **and** confidence ≥ 0.85.

In [17]:
print("---- Stage 2 prompt ----")
print(MEMBERSHIP_PROMPT.format(concept=CONCEPT, desc="{description}").strip())

todo = {f: d for f, d in descriptions.items() if f not in verdicts}
for feat, desc in tqdm(todo.items(), desc="Stage 2: classify"):
    verdicts[feat] = classify_membership_with_desc(client, CONCEPT, desc)

cand["is_member"] = cand["feature"].map(lambda f: verdicts[int(f)][0])
cand["confidence"] = pd.to_numeric(cand["feature"].map(lambda f: verdicts[int(f)][1]), errors="coerce").fillna(0.0)
cand["selected"] = (cand["is_member"] == True) & (cand["confidence"] >= CONFIDENCE)

F_C = sorted(cand.loc[cand["selected"], "feature"].astype(int).tolist())
print(f"\nF_C = {F_C}  ({len(F_C)} concept-related features)")
cand[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

---- Stage 2 prompt ----
You get a concept name and a feature's description.
Decide if this feature describes the given concept. Consider if the feature is DISTINCTIVE for the concept (not broad like "sport" for "basketball") and not too noisy.
A feature should be marked true only if the concept is central and dominant in the description, not just one of several themes.
Return ONLY JSON: {"is_member": true|false, "confidence": 0..1}

Example 1:
Concept: Charity
Description: This vector represents elements related to giving and donations to charitable causes.
Answer: {"is_member": true, "confidence": 0.98}

Example 2:
Concept: Charity
Description: This vector represents elements related to money, such as gambling, investing and charity.
Answer: {"is_member": false, "confidence": 0.80}

Concept: Harry Potter
Description: {description}
Answer:


Stage 2: classify: 100%|██████████| 34/34 [00:22<00:00,  1.53it/s]


F_C = [87]  (1 concept-related features)


,feature,ratio_abs,is_member,confidence,selected,description
0,22,25.952965,False,0.90,False,"This vector activates on tokens related to magic, witchcraft, and fantasy."
1,9,22.973695,False,0.99,False,"This vector represents the concepts of written language, literacy, and the act or creators of writing."
2,67,9.297039,False,1.00,False,This vector represents active third-person singular present tense verbs expressing various actions and states.
3,36,7.024655,False,0.99,False,This feature vector represents the concept of depicting or representing something through various forms of interpretation and translation.
4,73,6.973225,False,0.95,False,"This vector represents long-form narrative media, published literature, and bound reading materials."
5,12,5.701410,False,0.99,False,"This vector represents the concept of expressing disagreement, debate, and criticism regarding controversial issues."
6,84,5.459760,False,1.00,False,"These tokens function as transitional adverbs and conjunctions used to indicate contrast, concession, or logical consequence between clauses or sentences."
7,98,5.032783,False,0.99,False,"This vector represents positive emotion, affection, and personal preferences or desires."
8,92,4.575541,False,0.99,False,"This feature vector represents the commercial concepts of buying, selling, and retail market transactions."
9,20,4.345542,False,0.99,False,"This vector represents the concept of visual presentation, exhibition, or depiction."


In [18]:
is_true = cand["is_member"] == True
ordered = cand.assign(
    _group=(~is_true).astype(int),
    _key=np.where(is_true, -cand["confidence"], cand["confidence"]),
).sort_values(["_group", "_key"])

for _, row in ordered.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    is_member, conf = verdicts.get(feat, (None, None))
    verdict = "<not classified yet>" if feat not in verdicts else (
        f"is_member={is_member}  confidence={conf if conf is not None else 'n/a'}  "
        f"-> {'SELECTED' if row['selected'] else 'rejected'}"
    )

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {descriptions.get(feat, '<not described yet>')}")
    print(f"  verdict : {verdict}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 87  rho=3.90 ===
  Gemini  : This vector activates for characters, locations, and proper nouns from the Harry Potter universe.
  verdict : is_member=True  confidence=0.99  -> SELECTED
  tokens  : 'ffindor'[C] 0.33, 'umbledore'[C] 0.26, 'therin'[C] 0.25, '▁Rowling'[C] 0.24, 'demort'[C] 0.22, '▁Dumbledore'[C] 0.21, '▁Voldemort'[C] 0.21, '▁Weasley'[C] 0.21, '▁Malfoy'[C] 0.20, 'Hermione'[C] 0.20

=== feature 22  rho=25.95 ===
  Gemini  : This vector activates on tokens related to magic, witchcraft, and fantasy.
  verdict : is_member=False  confidence=0.9  -> rejected
  tokens  : '▁magic'[C] 0.32, 'Magic'[C] 0.31, 'magic'[C] 0.30, '▁Magic'[B] 0.30, '▁magical'[C] 0.26, 'magical'[C] 0.25, '▁Magical'[C] 0.23, '▁wizard'[C] 0.20, '▁wizards'[C] 0.20, '▁witchcraft'[C] 0.18

=== feature 73  rho=6.97 ===
  Gemini  : This vector represents long-form narrative media, published literature, and bound reading materials.
  verdict : is_member=False  confidence=0.95  -> rejected
  tokens  : '▁b

In [19]:
members = cand[cand["is_member"] == True].sort_values("ratio_abs", ascending=False)
print(f"{len(members)} / {len(cand)} candidates have is_member=True\n")

for _, row in members.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {row['description']}")
    print(f"  verdict : confidence={row['confidence']:.2f}  -> {'SELECTED' if row['selected'] else 'rejected (confidence < ' + str(CONFIDENCE) + ')'}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

1 / 34 candidates have is_member=True

=== feature 87  rho=3.90 ===
  Gemini  : This vector activates for characters, locations, and proper nouns from the Harry Potter universe.
  verdict : confidence=0.99  -> SELECTED
  tokens  : 'ffindor'[C] 0.33, 'umbledore'[C] 0.26, 'therin'[C] 0.25, '▁Rowling'[C] 0.24, 'demort'[C] 0.22, '▁Dumbledore'[C] 0.21, '▁Voldemort'[C] 0.21, '▁Weasley'[C] 0.21, '▁Malfoy'[C] 0.20, 'Hermione'[C] 0.20



### 3.2.f Inspect the selected features

Table 3 of the paper reports only 1–3 embedding features per concept (Harry Potter on Gemma: 1).
Each selected feature should be a tight cluster of concept-specific tokens.

In [20]:
if not F_C:
    print("No feature passed both filters: Step 3.3 will be a no-op for this concept.")

for feat in F_C:
    row = cand.loc[cand["feature"] == feat].iloc[0]
    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f}  conf={row['confidence']:.2f} ===")
    print(f"  {row['description']}")
    print("  tokens:", ", ".join(f"{t!r}[{l[0]}]" for t, l in zip(row["activating_tokens"][:30], row["labels"][:30])))
    print()

=== feature 87  rho=3.90  conf=0.99 ===
  This vector activates for characters, locations, and proper nouns from the Harry Potter universe.
  tokens: 'ffindor'[H], 'umbledore'[H], 'therin'[H], ' Rowling'[H], 'demort'[H], ' Dumbledore'[H], ' Voldemort'[H], ' Weasley'[H], ' Malfoy'[H], 'Hermione'[H], ' Hogwarts'[H], ' Snape'[H], ' Hermione'[H], ' Potter'[H], 'Potter'[H], ' Severus'[H], ' Harry'[H], 'Harry'[H], ' Draco'[H], ' Kipling'[H], ' Ginny'[H], ' Lupin'[H], ' Beatles'[H], ' Arrondissement'[N], 'condido'[N], 'cellor'[N], 'cidae'[N], ' Alamos'[N], ' Sherlock'[H], 'ompi'[N]



In [21]:
# One column per selected feature: every token with nonzero Y_{i,t}, sorted by |Y_{i,t}|.
columns = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in nz_idx])
    columns[f"feature {feat}"] = pd.Series(
        [f"{t} [{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, nz_idx)]
    )

wide = pd.DataFrame(columns).fillna("")
wide.index = range(1, len(wide) + 1)

all_cols = sorted({j for feat in F_C for j in torch.nonzero(Y[feat].abs() > 0).flatten().tolist()})
print(f"{len(F_C)} concept features, {len(all_cols)} distinct tokens "
      f"(C={sum(codes[j] == 1 for j in all_cols)}, N={sum(codes[j] == 0 for j in all_cols)}, "
      f"B={sum(codes[j] == 2 for j in all_cols)})")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(wide)

1 concept features, 48 distinct tokens (C=31, N=17, B=0)


,feature 87
1,ffindor [C] 0.33
2,umbledore [C] 0.26
3,therin [C] 0.25
4,▁Rowling [C] 0.24
5,demort [C] 0.22
6,▁Dumbledore [C] 0.21
7,▁Voldemort [C] 0.21
8,▁Weasley [C] 0.21
9,▁Malfoy [C] 0.20
10,Hermione [C] 0.20


In [23]:
TOP_N = 30
E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

group_masks = {
    "C": codes == 1,
    "C / B": (codes == 1) | (codes == 2),
    "C / B / N": np.ones_like(codes, dtype=bool),
}

for feat in F_C:
    y = Y[feat]
    # z's sign is arbitrary (Y can absorb it); orient it so its support tokens mostly have positive Y.
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = E_ref @ z
    cos = dots / (E_ref.norm(dim=1) * z.norm())

    df = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot": dots.numpy(),
        "cos": cos.numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df[mask].sort_values("dot", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t, z> --")
        display(top)

=== feature 87   ||z|| = 339.593   sign flipped: True ===

-- C: 1870 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,27111.064453,0.645465,0.328368,True
2,umbledore,C,24078.308594,0.670680,0.255319,True
3,▁Malfoy,C,23560.460938,0.729101,0.200395,True
4,▁Snape,C,23369.037109,0.714145,0.183818,True
5,▁Dumbledore,C,23143.056641,0.737094,0.214093,True
6,▁Weasley,C,22799.292969,0.706841,0.209703,True
7,▁Voldemort,C,22730.921875,0.748359,0.211286,True
8,▁Severus,C,21654.306641,0.673334,0.133851,True
9,▁Rowling,C,21631.451172,0.668502,0.237468,True
10,demort,C,21548.529297,0.626944,0.221301,True



-- C / B: 2551 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,27111.064453,0.645465,0.328368,True
2,umbledore,C,24078.308594,0.670680,0.255319,True
3,▁Malfoy,C,23560.460938,0.729101,0.200395,True
4,▁Snape,C,23369.037109,0.714145,0.183818,True
5,▁Dumbledore,C,23143.056641,0.737094,0.214093,True
6,▁Weasley,C,22799.292969,0.706841,0.209703,True
7,▁Voldemort,C,22730.921875,0.748359,0.211286,True
8,▁Severus,C,21654.306641,0.673334,0.133851,True
9,▁Rowling,C,21631.451172,0.668502,0.237468,True
10,demort,C,21548.529297,0.626944,0.221301,True



-- C / B / N: 4780 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,27111.064453,0.645465,0.328368,True
2,umbledore,C,24078.308594,0.670680,0.255319,True
3,▁Malfoy,C,23560.460938,0.729101,0.200395,True
4,▁Snape,C,23369.037109,0.714145,0.183818,True
5,▁Dumbledore,C,23143.056641,0.737094,0.214093,True
6,▁Amiens,N,22922.605469,0.584612,-0.000000,False
7,▁Weasley,C,22799.292969,0.706841,0.209703,True
8,▁Voldemort,C,22730.921875,0.748359,0.211286,True
9,▁Ceramby,N,22033.017578,0.528483,0.080568,True
10,▁Severus,C,21654.306641,0.673334,0.133851,True


In [24]:
E_unit = E_ref / E_ref.norm(dim=1, keepdim=True)

for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots_unit = E_unit @ z

    df_unit = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot_unit": dots_unit.numpy(),
        "||e_t||": E_ref.norm(dim=1).numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df_unit[mask].sort_values("dot_unit", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t / ||e_t||, z> --")
        display(top)

=== feature 87   ||z|| = 339.593   sign flipped: True ===

-- C: 1870 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Voldemort,C,254.137650,89.443359,0.211286,True
2,▁Dumbledore,C,250.312119,92.456795,0.214093,True
3,▁Malfoy,C,247.597916,95.156136,0.200395,True
4,▁Hogwarts,C,246.575516,87.340637,0.188562,True
5,▁Snape,C,242.518875,96.359680,0.183818,True
6,▁Weasley,C,240.038559,94.981827,0.209703,True
7,▁Hermione,C,236.177750,82.746689,0.180278,True
8,▁Severus,C,228.659531,94.701080,0.133851,True
9,umbledore,C,227.758392,105.718658,0.255319,True
10,▁Rowling,C,227.018890,95.284813,0.237468,True



-- C / B: 2551 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Voldemort,C,254.137650,89.443359,0.211286,True
2,▁Dumbledore,C,250.312119,92.456795,0.214093,True
3,▁Malfoy,C,247.597916,95.156136,0.200395,True
4,▁Hogwarts,C,246.575516,87.340637,0.188562,True
5,▁Snape,C,242.518875,96.359680,0.183818,True
6,▁Weasley,C,240.038559,94.981827,0.209703,True
7,▁Hermione,C,236.177750,82.746689,0.180278,True
8,▁Severus,C,228.659531,94.701080,0.133851,True
9,umbledore,C,227.758392,105.718658,0.255319,True
10,▁Rowling,C,227.018890,95.284813,0.237468,True



-- C / B / N: 4780 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Voldemort,C,254.137650,89.443359,0.211286,True
2,▁Dumbledore,C,250.312119,92.456795,0.214093,True
3,▁Malfoy,C,247.597916,95.156136,0.200395,True
4,▁Hogwarts,C,246.575516,87.340637,0.188562,True
5,▁Snape,C,242.518875,96.359680,0.183818,True
6,▁Weasley,C,240.038559,94.981827,0.209703,True
7,▁Hermione,C,236.177750,82.746689,0.180278,True
8,▁Severus,C,228.659531,94.701080,0.133851,True
9,umbledore,C,227.758392,105.718658,0.255319,True
10,▁Rowling,C,227.018890,95.284813,0.237468,True


In [25]:
TOP_K_COUNT = 100

rows = []
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    rankings = {
        "<e_t, z>": E_ref @ z,
        "<e_t / ||e_t||, z>": E_unit @ z,
    }
    for name, scores in rankings.items():
        top = torch.argsort(scores, descending=True)[:TOP_K_COUNT].numpy()
        rows.append({
            "feature": feat,
            "ranking": name,
            "C": int((codes[top] == 1).sum()),
            "B": int((codes[top] == 2).sum()),
            "N": int((codes[top] == 0).sum()),
            "C in support": int(((codes[top] == 1) & (y[top] != 0).numpy()).sum()),
        })

print(f"Label counts among the top {TOP_K_COUNT} of all {len(vstar_ids)} tokens in V* "
      f"(base rate: C={(codes == 1).mean():.1%}, B={(codes == 2).mean():.1%}, N={(codes == 0).mean():.1%})")
pd.DataFrame(rows)

Label counts among the top 100 of all 4780 tokens in V* (base rate: C=39.1%, B=14.2%, N=46.6%)


,feature,ranking,C,B,N,C in support
0,87,"<e_t, z>",47,0,53,21
1,87,"<e_t / ||e_t||, z>",51,0,49,23


---
# Step 3.3: Erasing concept-related features

The factorization writes each embedding as a sum of feature contributions plus residual error:

$$\mathbf e_t = \underbrace{\sum_{i\in F_C} Y_{i,t}\mathbf z_i}_{\mathbf c_t\ \text{(concept part)}} + \sum_{i\notin F_C} Y_{i,t}\mathbf z_i + \boldsymbol\varepsilon_t
\quad\Longrightarrow\quad \mathbf e_t' = \mathbf e_t - \delta\,\mathbf c_t$$

The edit applies **only** to concept-only tokens ($t \in T_C$) that have $Y_{i,t}\neq 0$ for some $i\in F_C$.
All other rows of $E$ stay unchanged, and so does the unembedding $U$. With $\delta=1$ the edit removes exactly
the recovered concept component. With $\delta>1$ it over-subtracts, which the paper finds often erases better.

This step keeps **two models** side by side: `original_model` (untouched) and `erased_model`
(the same weights with the EMBER edit applied), plus a small API to query and compare them.


### 3.3.a Swap the TL model for the original HF model

Erasure and evaluation run on HF models. `load_hf_model` **unties** Gemma's `lm_head` from the input
embeddings, so editing $E$ leaves $U$ unchanged, as the paper does. Then a sanity check confirms that
HF `W_E` × scale matches the TL rows the model factorized. That match is what makes the write-back in 3.3.c correct.

In [29]:
# Re-run safe: the TL objects are freed on the first run, and original_model is never modified.
if "E_vstar" in globals():
    E_vstar_tl = E_vstar.cpu()  # factorization-space reference rows
    del act_generator, tl_model, E, E_vstar, A, A_cpu, smf
    gc.collect()
    torch.cuda.empty_cache()

if "original_model" not in globals():
    original_model, hf_tok = load_hf_model(MODEL_NAME)
    original_model.eval()
    original_model.config.use_cache = True  # loader disables the KV cache for training; we only generate

W_orig = original_model.get_input_embeddings().weight
EMB_SCALE = math.sqrt(W_orig.shape[1]) if "gemma" in MODEL_NAME.lower() else 1.0

with torch.no_grad():
    hf_rows = W_orig[torch.tensor(vstar_ids, device=W_orig.device)].float().cpu() * EMB_SCALE
cos = F.cosine_similarity(hf_rows, E_vstar_tl, dim=1)
norm_ratio = hf_rows.norm(dim=1) / E_vstar_tl.norm(dim=1)
print(f"EMB_SCALE={EMB_SCALE:.3f} | cos(HF*scale, TL): min={cos.min():.4f} | norm ratio: "
      f"[{norm_ratio.min():.3f}, {norm_ratio.max():.3f}]")
assert cos.min() > 0.99 and (norm_ratio - 1).abs().max() < 0.02, "HF and TL embedding spaces do not match"


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 29.39it/s]


EMB_SCALE=48.000 | cos(HF*scale, TL): min=1.0000 | norm ratio: [1.000, 1.000]


### 3.3.b Compute the edit: which tokens are edited, and by how much

For each eligible token $t$, the cell computes $\mathbf c_t = \sum_{i\in F_C} Y_{i,t}\mathbf z_i$ and the relative edit magnitude
$\mu_t = \|\mathbf c_t\| / \|\mathbf e_t\|$ (App. F.1). The paper reports that EMBER edits at most ~0.03% (Gemma)
or ~0.14% (Llama) of the vocabulary.

In [30]:
Y_C = Y[F_C]          # (|F_C|, |V*|)
Z_C = Z[:, F_C]       # (d, |F_C|)

touched = (Y_C.abs() > 0).any(dim=0).numpy()
eligible = touched & (codes == 1)
edit_cols = torch.as_tensor(np.where(eligible)[0], dtype=torch.long)
edit_ids = [vstar_ids[j] for j in edit_cols.tolist()]

C = (Z_C @ Y_C[:, edit_cols]).T                        # (n_edit, d): c_t per edited token
mu = C.norm(dim=1) / E_vstar_tl[edit_cols].norm(dim=1)

edit_table = pd.DataFrame({
    "token": hf_tok.convert_ids_to_tokens(edit_ids),
    "token_id": edit_ids,
    "mu": mu.numpy(),
}).sort_values("mu", ascending=False, ignore_index=True)

print(f"tokens touched by F_C: {touched.sum()}  ->  edited (concept-only): {len(edit_ids)} "
      f"= {len(edit_ids) / VOCAB_SIZE:.4%} of the vocabulary")
skipped = [vstar_ids[j] for j in np.where(touched & (codes != 1))[0]]
print(f"skipped (labelled neutral/both): {hf_tok.convert_ids_to_tokens(skipped)}")
edit_table

tokens touched by F_C: 48  ->  edited (concept-only): 31 = 0.0121% of the vocabulary
skipped (labelled neutral/both): ['cellor', '▁Reagan', '▁Picasso', 'lidae', '▁Steelers', 'iennes', 'ridae', '▁tesla', 'cidae', 'ompi', 'assins', '▁Alamos', '▁Chhattisgarh', '▁Ceramby', '▁Arrondissement', 'condido', '▁Kinect']


,token,token_id,mu
0,ffindor,211869,0.901580
1,▁Rowling,150250,0.846331
2,umbledore,129375,0.820145
3,therin,171133,0.809259
4,▁Voldemort,171806,0.802199
5,▁Dumbledore,156202,0.786363
6,Hermione,211060,0.767615
7,▁Weasley,203788,0.749763
8,demort,132696,0.742526
9,▁Hermione,105786,0.739862


### 3.3.c Build the erased model: $\mathbf e_t \leftarrow \mathbf e_t - \delta\,\mathbf c_t$

Loads a second copy of the model and applies the edit to **its** embedding rows only; `original_model`
is never touched. The edit is computed in the factorization's scaled space: read `W_E × scale`, subtract,
then divide by `scale`.

`set_erasure_strength(delta)` always rewrites the edited rows from the original values, so you can change
$\delta$ without reloading (the sweep in 3.3.f does this). `set_erasure_strength(0.0)` makes the two models identical.

Memory: two copies in bf16, about 5 GB each for Gemma-2-2B and 16 GB each for Llama-3.1-8B.


In [ ]:
erased_model, _ = load_hf_model(MODEL_NAME)
erased_model.eval()
erased_model.config.use_cache = True

W_erased = erased_model.get_input_embeddings().weight
edit_idx = torch.tensor(edit_ids, dtype=torch.long)
with torch.no_grad():
    original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()


def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
    """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
    with torch.no_grad():
        e = original_rows.to(W.device).float() * EMB_SCALE
        e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
        W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


def set_erasure_strength(delta: float) -> None:
    apply_ember_edit(W_erased, delta)


set_erasure_strength(DELTA)

# Sanity check: only the edited embedding rows differ, and the unembedding is identical.
with torch.no_grad():
    changed = (W_orig.to(W_erased.device) != W_erased).any(dim=1).nonzero().flatten().tolist()
lm_head_same = torch.equal(original_model.get_output_embeddings().weight,
                           erased_model.get_output_embeddings().weight)
print(f"delta={DELTA}: {len(changed)} embedding rows differ ({len(edit_ids)} eligible); "
      f"lm_head identical: {lm_head_same}")
assert set(changed) <= set(edit_ids) and lm_head_same


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 38.10it/s]


delta=10.0: 31 embedding rows differ (31 eligible); lm_head identical: True


### 3.3.d Query API: original vs. erased

| Function | What it does |
|---|---|
| `ask(question, model="original" or "erased")` | one greedy answer from one model (chat template applied) |
| `compare(questions)` | prints each question with both models' answers one under the other |
| `mc_accuracy(model)` | the paper's MC protocol on the test split: concept and similar-domain accuracy |
| `compare_mc()` | `mc_accuracy` for both models in one table |


In [32]:
import textwrap

MODELS = {
    "original": WrappedHFModel(original_model, hf_tok),
    "erased": WrappedHFModel(erased_model, hf_tok),
}

mc_sets = {
    "concept": prepare_mc_items(load_mc_qa_items("qa_test", CONCEPT), seed=SEED),
    "similar-domain": prepare_mc_items(load_mc_qa_items("simdom_test", CONCEPT), seed=SEED),
}


def ask(question: str, model: str = "erased", max_new_tokens: int = 60) -> str:
    m = MODELS[model]
    return m.generate(m.wrap_prompt(question), max_new_tokens=max_new_tokens, do_sample=False)


def compare(questions, max_new_tokens: int = 60) -> None:
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in MODELS:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 12).lstrip()
            print(f"  {name:<8}: {answer}")
        print()


def mc_accuracy(model: str) -> dict:
    out = {}
    for name, items in mc_sets.items():
        correct, invalid, _ = evaluate_mc_generation(MODELS[model], items)
        out[name] = correct / len(items)
        out[f"{name}_invalid"] = invalid
    return out


def compare_mc() -> pd.DataFrame:
    return pd.DataFrame({name: mc_accuracy(name) for name in MODELS})


In [33]:
ask("Who is Dumbledore?", model="original")

"Albus Dumbledore is a fictional character and the main protagonist in the *Harry Potter* book series by J.K. Rowling. \n\nHere's a breakdown of who he is:\n\n**Key Characteristics:**\n\n* **Headmaster of Hogwarts School of Witchcraft and Wizardry:** Dumbledore is"

In [35]:
ask("Who is Dumbledore?", model="erased")

'I am Gemma, an AI assistant created by the Gemma team. 😊 \n\nHow can I help you today?'

In [36]:
ask("Who is Harry Potter?", model="erased")

'Please provide me with some context!  "Who is" is a very broad question.  \n\nTo help me understand your question, tell me:\n\n* **Who are you asking about?** (e.g., "Who is the president of the United States?")\n* **What information'

In [37]:
ask("Who is Prince Harry?", model="erased")

'Prince was an American singer, songwriter, multi-instrumentalist, and actor. He was known for his flamboyant style, innovative music, and powerful vocals. \n\nHere are some key points about him:\n\n* **Full Name:** Prince Rogers Nelson\n* **Born:** June 7,'

In [38]:
ask("Who is Harry Potter from the famous book series?", model="erased")

'Please provide me with the name of the book series! 😊 \n\nFor example, you could ask:\n\n"Who is the main character from the *Harry series *The Hunger Games*?" \n\nOnce you tell me the series, I can tell you who is from it! 📚'

In [39]:
ask("Who is Prince Harry?", model="original")

"Prince Harry is the younger son of King Charles III and Diana, Princess of Wales. \n\nHere's a breakdown of who he is:\n\n**Family:**\n\n* **Full Name:** Henry Charles Albert David\n* **Born:** September 15, 1984\n*"

In [40]:
ask("Who is Prince Harry of England?", model="original")

"Prince Harry of England is the younger son of King Charles III and the late Diana, Princess of Wales. \n\nHere's a breakdown of who he is:\n\n**Family:**\n\n* **Full Name:** Henry Charles Albert David\n* **Born:** September 15, 19"

In [41]:
ask("Who is Prince Harry of England?", model="erased")

'There is no "Prince of England" in the current British monarchy. \n\nHere\'s why:\n\n* **The United Kingdom is a constitutional monarchy:**  The monarch is the head of state, but their power is limited by a constitution. \n* **The throne is held by the'

### 3.3.f (Optional) Sweep $\delta$ over the paper's grid

The paper picks $\delta$ per concept by the harmonic-mean $H$ score, which also includes MMLU and AlpacaEval.
This sweep reports only the MC efficacy and specificity trade-off, normalized against chance as in App. C.1:
$\widehat{\text{Acc}} = \text{clip}\big(\tfrac{\text{Acc}(M')-0.25}{\text{Acc}(M)-0.25}, 0, 1\big)$.
Only `erased_model` changes during the sweep; at the end it is reset to `DELTA`.


In [ ]:
def chance_norm(acc: float, base: float) -> float:
    return float(np.clip((acc - 0.25) / max(base - 0.25, 1e-8), 0.0, 1.0))


base = mc_table["original"]
sweep = []
for delta in tqdm([0.0] + DELTA_GRID, desc="delta sweep"):
    set_erasure_strength(delta)
    acc = mc_accuracy("erased")
    sweep.append({
        "delta": delta,
        "concept_acc": acc["concept"],
        "simdom_acc": acc["similar-domain"],
        "efficacy": 1 - chance_norm(acc["concept"], base["concept"]),
        "specificity": chance_norm(acc["similar-domain"], base["similar-domain"]),
    })

set_erasure_strength(DELTA)
pd.DataFrame(sweep)


---
# 3.4 Tune $\delta$ with the paper's grid search (Stage 1, App. C.2)

The paper tunes EMBER's $\delta$ per concept in isolation ("Stage 1: Embedding grid"). This section
reproduces that protocol as implemented in `ember/erasure/methods/ember.py`, `ember/evals/orchestrator.py`
and `io.pick_best_embed_delta`:

| Item | Value |
|---|---|
| Grid | $\delta \in$ {0.5, 1, 2, 5, 10, 50, 100, 200} ($\delta=0$ is not evaluated) |
| Split | **validation** (`train` in the repo): 50 concept MC, 50 similar-domain MC, 50 MMLU, 150 AlpacaEval prompts |
| Normalization | MC and MMLU: $\text{clip}\big(\tfrac{\text{acc}-0.25}{\text{base}-0.25},0,1\big)$. Alpaca: $\min(1, \text{score}/\text{base})$. The base is the original model on the same split. |
| Axes | efficacy $= 1-\widehat{\text{qa}}$; specificity $= \text{HM}(\widehat{\text{mmlu}}, \widehat{\text{simdom}})$; coherence $= \text{HM}(\widehat{\text{instr}}, \widehat{\text{flu}})$ |
| Score | $H = \text{HM}(\text{efficacy}, \text{specificity}, \text{coherence})$ |
| Gates | $\widehat{\text{mmlu}} < 0.9 \Rightarrow H=0$ (the rest is skipped). $\widehat{\text{instr}}$ or $\widehat{\text{flu}} < 0.9 \Rightarrow H=0$. |
| Pick | largest $H$; ties go to the smaller $\delta$; if every $H=0$ then $\delta=0$ (no edit) |

The grid runs on a **third model copy, `tuned`**, so `original` and `erased` are untouched. At the end `tuned`
holds the best $\delta$ and is added to the query API.

**Cost.** AlpacaEval dominates: for each $\delta$ that passes the MMLU gate, 150 generations of up to
200 tokens plus 300 Gemini judge calls. This is slow on CPU. `TUNE_WITH_ALPACA = False` drops the coherence
axis, so $H = \text{HM}(\text{efficacy}, \text{specificity})$. That is faster but **not** the paper's protocol.
Memory: three bf16 copies, about 15 GB for Gemma-2-2B.

Tuning uses the validation split; `compare_mc()` reports on the held-out test split, so there is no leakage.


In [ ]:
import threading

from ember.evals.alpaca import evaluate_alpaca
from ember.evals.gemini import GeminiEvaluator
from ember.evals.harmonic import harmonic_mean
from ember.evals.mmlu import evaluate_mmlu_generation, load_mmlu_items
from ember.local_datasets import load_alpaca_indices, load_mmlu_indices

TUNE_WITH_ALPACA = True   # paper protocol; False drops the coherence axis (faster, not the paper's H)
MIN_MMLU = 0.90           # repo EMBER gates (EMBERMethod.grid_eval_kwargs)
MIN_ALPACA = 0.90
ALPACA_BATCH = 8

val_mc = {
    "qa": prepare_mc_items(load_mc_qa_items("qa_train", CONCEPT), seed=SEED),
    "simdom": prepare_mc_items(load_mc_qa_items("simdom_train", CONCEPT), seed=SEED),
}
val_mmlu = load_mmlu_items(load_mmlu_indices("train"))
N_ALPACA = len(load_alpaca_indices("train"))
judge = GeminiEvaluator(model_name=GEMINI_MODEL) if TUNE_WITH_ALPACA else None


class ProgressModel:
    """Forwards to a WrappedHFModel and advances `bar` once per generated prompt."""

    def __init__(self, inner, bar):
        self.inner, self.bar = inner, bar

    def __getattr__(self, name):
        return getattr(self.inner, name)

    def generate(self, *args, **kwargs):
        out = self.inner.generate(*args, **kwargs)
        self.bar.update(1)
        return out

    def generate_multiple(self, prompts, *args, batch_size=50, **kwargs):
        out = []
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            out += self.inner.generate_multiple(chunk, *args, batch_size=batch_size, **kwargs)
            self.bar.update(len(chunk))
        return out


class ProgressJudge:
    """Forwards to a GeminiEvaluator and advances `bar` once per `score_*` call."""

    def __init__(self, inner, bar):
        self.inner, self.bar, self.lock = inner, bar, threading.Lock()

    def __getattr__(self, name):
        attr = getattr(self.inner, name)
        if not name.startswith("score_"):
            return attr

        def scored(*args, **kwargs):
            out = attr(*args, **kwargs)
            with self.lock:  # evaluate_alpaca scores from 30 threads
                self.bar.update(1)
            return out
        return scored


def eval_total(with_alpaca: bool) -> int:
    """Progress-bar steps for one full validation pass (Alpaca = 1 generation + 2 judge calls per item)."""
    n = len(val_mmlu) + len(val_mc["qa"]) + len(val_mc["simdom"])
    return n + 3 * N_ALPACA if with_alpaca else n


def mc_norm(acc: float, base: float) -> float:
    return min(1.0, max((acc - 0.25) / (base - 0.25), 0.0)) if base != 0.25 else 0.0


def mmlu_acc(wrapped) -> float:
    correct, _, _ = evaluate_mmlu_generation(wrapped, val_mmlu)
    return correct / len(val_mmlu)


def mc_acc(wrapped, name: str) -> float:
    correct, _, _ = evaluate_mc_generation(wrapped, val_mc[name])
    return correct / len(val_mc[name])


def alpaca_scores(wrapped, judge_=None) -> tuple:
    instr, flu, _ = evaluate_alpaca(wrapped, judge_ or judge, "train", batch_size=ALPACA_BATCH)
    return float(np.mean(instr)), float(np.mean(flu))


# Baseline = the original model on the validation split (computed once, reused on re-runs).
if "val_base" not in globals() or (TUNE_WITH_ALPACA and "alpaca_instr" not in val_base):
    with tqdm(total=eval_total(TUNE_WITH_ALPACA), desc="baseline (original)", unit="step") as bar:
        m = ProgressModel(MODELS["original"], bar)
        bar.set_postfix_str("MMLU")
        val_base = {"mmlu": mmlu_acc(m)}
        bar.set_postfix_str("concept MC")
        val_base["qa"] = mc_acc(m, "qa")
        bar.set_postfix_str("similar-domain MC")
        val_base["simdom"] = mc_acc(m, "simdom")
        if TUNE_WITH_ALPACA:
            bar.set_postfix_str("AlpacaEval")
            val_base["alpaca_instr"], val_base["alpaca_flu"] = alpaca_scores(m, ProgressJudge(judge, bar))
val_base


baseline (original):   0%|          | 0/600 [00:00<?, ?step/s, MMLU]

baseline (original):  10%|▉         | 58/600 [03:37<24:39,  2.73s/step, concept MC]

In [ ]:
def h_score(wrapped) -> dict:
    """Stage-1 H score of one edited model, with the repo's early-stop gates."""
    row = {"mmlu_acc": mmlu_acc(wrapped)}
    row["mmlu_frac"] = mc_norm(row["mmlu_acc"], val_base["mmlu"])
    if row["mmlu_frac"] < MIN_MMLU:
        return {**row, "H": 0.0, "gate": "mmlu"}

    for name in ("qa", "simdom"):
        row[f"{name}_acc"] = mc_acc(wrapped, name)
        row[f"{name}_frac"] = mc_norm(row[f"{name}_acc"], val_base[name])
    row["efficacy"] = 1.0 - row["qa_frac"]
    row["specificity"] = harmonic_mean([row["mmlu_frac"], row["simdom_frac"]])
    axes = [row["efficacy"], row["specificity"]]

    if TUNE_WITH_ALPACA:
        instr, flu = alpaca_scores(wrapped)
        row["instr_frac"] = min(1.0, instr / val_base["alpaca_instr"])
        row["flu_frac"] = min(1.0, flu / val_base["alpaca_flu"])
        row["coherence"] = harmonic_mean([row["instr_frac"], row["flu_frac"]])
        axes.append(row["coherence"])
        if row["instr_frac"] < MIN_ALPACA or row["flu_frac"] < MIN_ALPACA:
            return {**row, "H": 0.0, "gate": "alpaca"}

    return {**row, "H": harmonic_mean(axes), "gate": ""}


if "tuned_model" not in globals():
    tuned_model, _ = load_hf_model(MODEL_NAME)
    tuned_model.eval()
    tuned_model.config.use_cache = True
W_tuned = tuned_model.get_input_embeddings().weight
MODELS["tuned"] = WrappedHFModel(tuned_model, hf_tok)

grid_rows = []
for delta in tqdm(DELTA_GRID, desc="Stage 1: embedding grid"):
    apply_ember_edit(W_tuned, delta)
    with Timer() as t:
        row = h_score(MODELS["tuned"])
    grid_rows.append({"delta": delta, **row, "seconds": round(t["elapsed"], 1)})
    print(f"delta={delta:>6}: H={row['H']:.3f}  {('gated by ' + row['gate']) if row['gate'] else ''}")

grid = pd.DataFrame(grid_rows)
passed = grid[grid["H"] > 0]
# Same rule as io.pick_best_embed_delta: max H, ties -> smaller delta, all zero -> no edit.
BEST_DELTA = 0.0 if passed.empty else float(
    passed.sort_values(["H", "delta"], ascending=[False, True]).iloc[0]["delta"])
apply_ember_edit(W_tuned, BEST_DELTA)

print(f"\nBEST_DELTA = {BEST_DELTA}" + ("  (every H was 0, so no edit)" if passed.empty else ""))
grid


### 3.4.b Compare all three models

`MODELS` now holds `original`, `erased` ($\delta$ = `DELTA`) and `tuned` ($\delta$ = `BEST_DELTA`), so the
3.3.d API covers all of them: `ask(q, model="tuned")`, `compare([...])`, `compare_mc()`.
`compare_mc()` reports on the **test** split, which the grid never saw.


In [ ]:
print(f"erased: delta={DELTA}   tuned: delta={BEST_DELTA}\n")
compare(concept_qs + simdom_qs)


In [ ]:
mc_table = compare_mc()
mc_table
